# cli

> The CLI driver — the workflow core's first (and currently only) frontend.

Ships in-package as the `cjm-transcription-core` console script so the driver can never skew from the core. GUI presentation drivers come later and consume the same `pipeline` module; they never reimplement it (CLI-first / headless-core principle).

Prerequisite runtime (once, from the repo root):
```bash
cjm-ctl --cjm-config cjm.yaml setup-runtime
cjm-ctl --cjm-config cjm.yaml install-all --plugins plugins_test.yaml --force
```
Then e.g.:
```bash
cjm-transcription-core run path/to/audio.mp3 --yes
cjm-transcription-core run ep1.mp3 ep2.mp3 --transcriber cjm-transcription-plugin-voxtral-hf
```

In [ ]:
#| default_exp cli

In [ ]:
#| export
import argparse
import asyncio
import logging
from pathlib import Path
from typing import List, Optional

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue

from cjm_transcription_core.models import PipelineConfig
from cjm_transcription_core.pipeline import run_pipeline

logger = logging.getLogger(__name__)

In [ ]:
#| export
def build_parser() -> argparse.ArgumentParser:  # Configured CLI parser
    """Build the CLI parser (subcommands: run)."""
    parser = argparse.ArgumentParser(
        prog="cjm-transcription-core",
        description="Headless transcription pipeline: VAD -> segment -> convert -> transcribe.",
    )
    sub = parser.add_subparsers(dest="command", required=True)

    run = sub.add_parser("run", help="Run the pipeline over one or more audio files")
    run.add_argument("audio", nargs="+", help="Source audio file path(s), in order")
    run.add_argument("--manifests-dir", default=".cjm/manifests", help="Capability manifests directory")
    run.add_argument("--transcriber", default="cjm-transcription-plugin-whisper", help="Transcription capability name")
    run.add_argument("--vad-plugin", default="cjm-media-plugin-silero-vad", help="VAD capability name")
    run.add_argument("--ffmpeg-plugin", default="cjm-media-plugin-ffmpeg", help="Convert/segment capability name")
    run.add_argument("--max-segment-duration", type=float, default=300.0, help="Wall-clock cap per segment in seconds")
    run.add_argument("--sample-rate", type=int, default=16000, help="Model-input sample rate")
    run.add_argument("--channels", type=int, default=1, help="Model-input channel count")
    run.add_argument("--force", action="store_true", help="Bypass capability-side caches (VAD + transcription)")
    run.add_argument("-y", "--yes", action="store_true", help="Auto-accept HITL seams (headless mode)")
    run.add_argument("--output", default=None, help="Run-manifest output path (default: runs/<run_id>.json)")
    run.add_argument("-v", "--verbose", action="store_true", help="DEBUG-level logging")
    return parser

In [ ]:
#| export
def load_capabilities(
    manager: PluginManager,   # Freshly constructed manager
    instance_ids: List[str],  # Capability names to load (default instances)
) -> None:
    """Discover manifests + load each requested capability (default instance)."""
    manager.discover_manifests()
    discovered = {m.name: m for m in manager.discovered}
    for iid in instance_ids:
        meta = discovered.get(iid)
        if meta is None:
            raise SystemExit(
                f"capability {iid!r} not found in manifests "
                f"(discovered: {sorted(discovered)}) — run cjm-ctl install-all first"
            )
        if not manager.load_plugin(meta):
            raise SystemExit(f"failed to load capability {iid!r}")
        logger.info(f"loaded {iid}")

In [ ]:
#| export
async def run_command(
    args: argparse.Namespace,  # Parsed CLI arguments for the `run` subcommand
) -> int:  # Process exit code (0 = all sources completed)
    """Execute the `run` subcommand: full pipeline over the given audio files."""
    cfg = PipelineConfig(
        vad_plugin=args.vad_plugin,
        ffmpeg_plugin=args.ffmpeg_plugin,
        transcriber_plugin=args.transcriber,
        max_segment_duration=args.max_segment_duration,
        sample_rate=args.sample_rate,
        channels=args.channels,
        force=args.force,
        assume_yes=args.yes,
    )
    sources = [str(Path(p).resolve()) for p in args.audio]
    missing = [s for s in sources if not Path(s).exists()]
    if missing:
        raise SystemExit(f"missing audio file(s): {missing}")

    manager = PluginManager(search_paths=[Path(args.manifests_dir)])
    instance_ids = [cfg.ffmpeg_plugin, cfg.vad_plugin, cfg.transcriber_plugin]
    load_capabilities(manager, instance_ids)

    queue = JobQueue(deps=manager)
    await queue.start()
    try:
        manifest = await run_pipeline(manager, queue, cfg, sources)
    finally:
        await queue.stop()
        for iid in instance_ids:
            try:
                manager.unload_plugin(iid)
            except Exception as e:  # Best-effort teardown; never mask the run's outcome
                logger.warning(f"unload {iid} failed: {e}")

    out = Path(args.output) if args.output else Path("runs") / f"{manifest.run_id}.json"
    manifest.save(out)
    done = sum(len(s.segments) for s in manifest.sources)
    print(f"run manifest: {out}")
    print(f"sources completed: {len(manifest.sources)}/{len(sources)}  segments transcribed: {done}")
    return 0 if len(manifest.sources) == len(sources) else 1

In [ ]:
#| export
def main(
    argv: Optional[List[str]] = None,  # Argument list override (None = sys.argv)
) -> int:  # Process exit code
    """CLI entry point (console script: `cjm-transcription-core`)."""
    args = build_parser().parse_args(argv)
    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s :: %(message)s",
    )
    if args.command == "run":
        return asyncio.run(run_command(args))
    raise SystemExit(f"unknown command: {args.command}")

In [ ]:
# Parser smoke checks (no plugins involved)
p = build_parser()
args = p.parse_args(["run", "a.mp3", "b.mp3", "--yes", "--max-segment-duration", "120"])
assert args.command == "run"
assert args.audio == ["a.mp3", "b.mp3"]
assert args.yes is True
assert args.max_segment_duration == 120.0
assert args.transcriber == "cjm-transcription-plugin-whisper"
print("cli parser checks OK")

cli parser checks OK
